# Lesson 6, one call at a time

Lesson 6 met three families of classifier on the same 1,200 pumps: k-nearest
neighbours, which stores the pumps and lets the nearest vote; Naive Bayes, which
multiplies one bell per reading; and the support vector machine, which takes the
widest margin and then changes the coordinates with a kernel. This notebook takes
each step on its own, on a few points where a pencil can follow it and on the
lesson's own pumps where the slides' numbers come from, so that the slide's number
and the call that makes it sit side by side.

**How to use it.** Each section is one *atom*. Before each code cell that teaches
something there is a question under *Your prediction, before you run the next cell*:
answer it first, then run. Notebook `00_the_scikit_learn_api.ipynb` explains `fit`,
`predict`, `predict_proba`, `decision_function` and `Pipeline`; this one assumes them.

Every atom names the slide it comes from, by title, and lesson 6's notebooks treat
each idea at full scale. The whole notebook runs in under a minute.

## Setup

In [1]:
# Everything used here ships with the course image:
# pip install numpy==1.26.4 pandas==2.1.1 scikit-learn==1.3.1

import sys
from functools import partial
from math import comb
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import (StratifiedKFold, cross_val_score, cross_validate,
                                     train_test_split)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Lesson 6's data: 1,200 pumps, vibration (Hz) and pressure (bar), faulty when the two
# readings together are too far from the design point, 42 Hz and 5.6 bar; 4% of the
# labels were flipped after the rule was applied.
sys.path.insert(0, str(Path.cwd().parent / "Lessons" / "06_knn_naive_bayes_svm" / "Notebooks"))
from pump_data import _standardised_radius, load_interacting, load_pumps, load_with_noise

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 120)

X, y = load_pumps()
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)   # the lesson's five folds
print(f"{len(X)} pumps, {int(y.sum())} faulty, columns: {list(X.columns)}")

1200 pumps, 735 faulty, columns: ['vibration_hz', 'pressure_bar']


## 1. One pump, its five nearest, and the vote

`fit` on a k-nearest-neighbours model stores the pumps; `kneighbors` returns the nearest ones and their distances; `predict` is their majority vote. The same split and train-only scaler as notebook 1, and the new pump at 46 Hz and 5.8 bar.

*From:* slides *k-nearest neighbours: the whole algorithm*; *k-NN, drawn: one new pump and its five nearest*; notebook 1, section 2.

### Your prediction, before you run the next cell

How many of the five nearest stored pumps will be faulty? And the one that is not: is it really healthy?

In [2]:
rng = np.random.default_rng(0)                      # notebook 1's split
test_index = rng.choice(len(X), 300, replace=False)
train_index = np.setdiff1d(np.arange(len(X)), test_index)
scaler = StandardScaler().fit(X.iloc[train_index])
knn = KNeighborsClassifier(n_neighbors=5).fit(scaler.transform(X.iloc[train_index]),
                                              y.iloc[train_index])
print(f"fit stored {knn.n_samples_fit_} pumps and nothing else to learn")

new_pump = pd.DataFrame([[46.0, 5.8]], columns=X.columns)
distance, position = knn.kneighbors(scaler.transform(new_pump))
stored = X.iloc[train_index].iloc[position[0]]
labels = y.iloc[train_index].iloc[position[0]]
for (index, row), d, label in zip(stored.iterrows(), distance[0], labels):
    radius = _standardised_radius(row["vibration_hz"], row["pressure_bar"])
    print(f"  pump {index}: {row['vibration_hz']:.2f} Hz, {row['pressure_bar']:.2f} bar, "
          f"distance {d:.2f}, label {'faulty' if label else 'healthy'}, "
          f"rule says {'faulty' if radius > 1 else 'healthy'}")
print(f"predict: {'faulty' if knn.predict(scaler.transform(new_pump))[0] else 'healthy'}")

fit stored 900 pumps and nothing else to learn
  pump 859: 46.46 Hz, 5.79 bar, distance 0.12, label faulty, rule says faulty
  pump 285: 45.60 Hz, 5.75 bar, distance 0.15, label faulty, rule says faulty
  pump 625: 45.37 Hz, 5.81 bar, distance 0.17, label faulty, rule says faulty
  pump 1058: 46.60 Hz, 5.83 bar, distance 0.17, label healthy, rule says faulty
  pump 1111: 45.39 Hz, 5.83 bar, distance 0.18, label faulty, rule says faulty
predict: faulty


**What to look at.** `fit` stored the 900 training pumps and learned nothing else. The five nearest are four faulty and one healthy, so `predict` says faulty. The healthy one, pump 1058 at 46.60 Hz and 5.83 bar, is a flipped label: the rule that generated the data calls it faulty, and the vote of five outvotes it.

**Try changing.** set `n_neighbors=1`: which pump decides now, and is the answer still faulty?

## 2. Scale first, or vibration picks the neighbours

The same pump and the same stored pumps, without the scaler: distances in raw Hz and bar. Count how many of the five nearest change.

*From:* slide *Scale first, or vibration decides everything*; handout section 2.1.

### Your prediction, before you run the next cell

Vibration spans 18 Hz and pressure 2.3 bar. Unscaled, will the five nearest be the same five pumps as in atom 1?

In [3]:
raw = KNeighborsClassifier(n_neighbors=5).fit(X.iloc[train_index], y.iloc[train_index])
raw_distance, raw_position = raw.kneighbors(new_pump)
raw_stored = X.iloc[train_index].iloc[raw_position[0]]
print(raw_stored.assign(distance=raw_distance[0].round(3)).to_string())
print(f"\nin common with the scaled five: {len(set(raw_stored.index) & set(stored.index))} of 5")
print(f"pressure spread among the unscaled five: {raw_stored['pressure_bar'].min():.2f} "
      f"to {raw_stored['pressure_bar'].max():.2f} bar")

      vibration_hz  pressure_bar  distance
728         46.114        5.6615     0.179
1032        45.846        5.9317     0.203
1148        45.791        5.9047     0.234
821         46.198        6.0389     0.310
739         45.656        5.7288     0.351

in common with the scaled five: 0 of 5
pressure spread among the unscaled five: 5.66 to 6.04 bar


**What to look at.** Without the scaler the five nearest are five different pumps: none in common with atom 1. They are all within half a hertz of 46 Hz, while their pressures run from 5.66 to 6.04 bar: a difference of 0.4 bar counts for less than a difference of 0.4 Hz, so vibration alone picks the neighbours.

**Try changing.** measure pressure in millibar (`X['pressure_bar'] * 1000`) and look at the unscaled neighbours again: now which reading decides?

## 3. Training accuracy at k = 1, and inside the folds

`cross_validate` with `return_train_score=True` scores each fold's model twice: on the 960 pumps it stores and on the 240 it has not seen.

*From:* slides *k sets the balance between bias and variance*; *Training against cross-validated accuracy, by k*; notebook 1, section 3.

### Your prediction, before you run the next cell

At k = 1, what is the training accuracy, exactly? At k = 401, will the training and cross-validated scores be far apart or close?

In [4]:
for k in (1, 5, 401):
    result = cross_validate(make_pipeline(StandardScaler(), KNeighborsClassifier(k)),
                            X, y, cv=folds, return_train_score=True)
    print(f"k = {k:>3}: training {result['train_score'].mean():.3f}   "
          f"cross-validated {result['test_score'].mean():.3f}   "
          f"per-fold training {result['train_score'].round(3)}")

k =   1: training 1.000   cross-validated 0.912   per-fold training [1. 1. 1. 1. 1.]


k =   5: training 0.954   cross-validated 0.944   per-fold training [0.952 0.955 0.954 0.954 0.955]


k = 401: training 0.713   cross-validated 0.708   per-fold training [0.718 0.72  0.701 0.71  0.716]


**What to look at.** At k = 1 the training accuracy is exactly 1.000 in every fold: each stored pump is its own nearest neighbour. At k = 5 the two scores are 0.954 and 0.944; at k = 401 they are 0.713 and 0.708, close together and both low: the model is too smooth everywhere, which is high bias, not overfitting.

**Try changing.** k = 101: are the two scores closer or further apart than at k = 5?

## 4. Distances stop varying

A separate experiment, with no pumps: 1,000 random points with n columns, each uniform on [0, 1], and for 60 new points the nearest distance divided by the farthest.

*From:* slides *Why: distances stop varying*; *With n random columns, the nearest point stops being near*; notebook 1, section 5.

### Your prediction, before you run the next cell

With 2 columns the ratio is about 0.02. With 100 columns, closer to 0.1, 0.4 or 0.7?

In [5]:
rng = np.random.default_rng(0)
for n in (2, 100):
    points = rng.random((1_000, n))
    ratios = []
    for query in rng.random((60, n)):
        d = np.sqrt(((points - query) ** 2).sum(axis=1))
        ratios.append(d.min() / d.max())
    d2 = ((points - rng.random(n)) ** 2).sum(axis=1)
    print(f"n = {n:>3}: nearest / farthest {np.mean(ratios):.3f};  "
          f"squared distance mean {d2.mean():.2f}, sd {d2.std():.2f}, sd/mean {d2.std() / d2.mean():.2f}")

n =   2: nearest / farthest 0.016;  squared distance mean 0.28, sd 0.22, sd/mean 0.79
n = 100: nearest / farthest 0.698;  squared distance mean 16.54, sd 1.80, sd/mean 0.11


**What to look at.** With 2 columns the nearest point is 0.016 of the way to the farthest; with 100 it is 0.698. The squared distance grows from about 0.28 to 16.5, but its spread grows far less, so relative to its size it shrinks from 0.79 to 0.11: every distance comes out about the same, and the nearest is barely nearer than a random point.

**Try changing.** n = 500: how close to 1 does the ratio get?

## 5. Choosing the columns: the obvious test fails

With 50 noise columns beside the two readings, keep two columns inside the pipeline. `f_classif` ranks a column by how far apart the class means are; `mutual_info_classif` by any dependence on the label.

*From:* slide *What the curse is, and is not*; notebook 1, section 5; handout section 3.3.

### Your prediction, before you run the next cell

Healthy and faulty pumps have almost the same mean vibration and pressure. Which chooser keeps the two real readings?

In [6]:
Xn, yn = load_with_noise(50)
for name, score_func in (("f_classif", f_classif),
                         ("mutual_info_classif", partial(mutual_info_classif, random_state=0))):
    chooser = SelectKBest(score_func, k=2).fit(StandardScaler().fit_transform(Xn), yn)
    kept = list(Xn.columns[chooser.get_support()])
    pipe = make_pipeline(StandardScaler(), SelectKBest(score_func, k=2), KNeighborsClassifier(15))
    print(f"{name:<20} keeps {kept}, cross-validated {cross_val_score(pipe, Xn, yn, cv=folds).mean():.3f}")
print(f"{'no selection':<20} cross-validated "
      f"{cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(15)), Xn, yn, cv=folds).mean():.3f}")

f_classif            keeps ['noise_024', 'noise_039'], cross-validated 0.562


mutual_info_classif  keeps ['vibration_hz', 'pressure_bar'], cross-validated 0.938
no selection         cross-validated 0.602


**What to look at.** `f_classif` keeps two noise columns, `noise_024` and `noise_039`, and scores 0.562, worse than keeping all 52 columns (0.602): healthy and faulty pumps have nearly the same mean vibration and pressure, so a test of the means finds nothing in them. `mutual_info_classif` keeps vibration and pressure and gets back 0.938, the score without noise.

**Try changing.** keep four columns, `k=4`, with `f_classif`: does it find the two readings now?

## 6. Ten numbers, and the 48 Hz pump by three ratios

`GaussianNB.fit` computes a share and, per class and reading, a mean and a variance. Then the pump at 48 Hz and 5.6 bar: the height of each bell, three ratios, and `predict_proba`.

*From:* slides *Training computes ten numbers*; *A pump at 48 Hz, 5.6 bar: there the faulty bell is 86 times higher*; notebook 2, section 3.

### Your prediction, before you run the next cell

The classes have the same centres. What separates them, then? And for this pump, will the three ratios give the same probability as `predict_proba`?

In [7]:
gnb = GaussianNB().fit(X, y)
sd = np.sqrt(gnb.var_)
print(pd.DataFrame({"share": gnb.class_prior_, "mean Hz": gnb.theta_[:, 0], "sd Hz": sd[:, 0],
                    "mean bar": gnb.theta_[:, 1], "sd bar": sd[:, 1]},
                   index=["healthy", "faulty"]).round(4).to_string())

def bell(value, mean, sd):
    return np.exp(-(value - mean) ** 2 / (2 * sd ** 2)) / (sd * np.sqrt(2 * np.pi))

h = [bell(48.0, gnb.theta_[c, 0], sd[c, 0]) for c in (0, 1)]
p = [bell(5.6, gnb.theta_[c, 1], sd[c, 1]) for c in (0, 1)]
odds = (gnb.class_prior_[1] / gnb.class_prior_[0]) * (h[1] / h[0]) * (p[1] / p[0])
print(f"\nvibration ratio {h[1] / h[0]:.1f}, pressure ratio {p[1] / p[0]:.3f} (healthy {p[0] / p[1]:.2f} times higher), "
      f"share ratio {gnb.class_prior_[1] / gnb.class_prior_[0]:.2f}")
print(f"odds {odds:.1f} to 1, so P(faulty) = {odds / (1 + odds):.3f}")
print(f"predict_proba: {gnb.predict_proba(pd.DataFrame([[48.0, 5.6]], columns=X.columns))[0].round(3)}")

          share  mean Hz   sd Hz  mean bar  sd bar
healthy  0.3875  41.9857  1.6945    5.6173  0.2187
faulty   0.6125  42.0151  4.6362    5.6159  0.5763

vibration ratio 86.4, pressure ratio 0.381 (healthy 2.63 times higher), share ratio 1.58
odds 52.0 to 1, so P(faulty) = 0.981
predict_proba: [0.019 0.981]


**What to look at.** The shares are 0.3875 and 0.6125; the centres are the same to a few hundredths; the standard deviations are not, 1.69 against 4.64 Hz and 0.22 against 0.58 bar: the widths separate the classes. At 48 Hz the faulty bell is 86.4 times the healthy one; at 5.6 bar the healthy bell is 2.63 times higher; the shares favour faulty 1.58 to 1. Together 86.4 × 1.58 ÷ 2.63 = 52 to 1, so P(faulty) = 52/53 = 0.981, the number `predict_proba` gives.

**Try changing.** the pump at the design point, 42 Hz and 5.6 bar: which ratio decides it now?

## 7. In logs: two totals and their gap

`predict_joint_log_proba` returns the log of share times densities for each class. The probability needs only the gap between the two.

*From:* slides *Logs turn the product into a sum*; *48 Hz, 5.6 bar: scores and their logs*; *Software uses only the gap between the logs*.

### Your prediction, before you run the next cell

The healthy total comes out near −8 and the faulty near −4. Is the faulty probability about 0.5, 0.9 or 0.98?

In [8]:
pump = pd.DataFrame([[48.0, 5.6]], columns=X.columns)
totals = gnb.predict_joint_log_proba(pump)[0]
gap = totals[1] - totals[0]
print(f"log totals: healthy {totals[0]:.3f}, faulty {totals[1]:.3f}, gap {gap:.3f}")
print(f"1 / (1 + e^-gap) = {1 / (1 + np.exp(-gap)):.3f};  e^gap = {np.exp(gap):.1f} to 1")
print(f"200 factors of 0.01: product {0.01 ** 200}, sum of logs {200 * np.log(0.01):.0f}")

log totals: healthy -8.095, faulty -4.144, gap 3.951
1 / (1 + e^-gap) = 0.981;  e^gap = 52.0 to 1
200 factors of 0.01: product 0.0, sum of logs -921


**What to look at.** The two log totals are −8.095 and −4.144, and only their gap, 3.951, matters: 1/(1 + e^(−3.951)) = 0.981, and e^3.951 = 52 is atom 6's odds. Two hundred factors of 0.01 make a product of exactly 0.0 in floating point, while their logs add to −921, an ordinary number: that is why software works in logs.

**Try changing.** add 100 to both totals: does the probability change?

## 8. Uncorrelated within each class, but not independent

The correlation of the two readings inside each class, then of their squared standardised distances from the centre.

*From:* slide *Uncorrelated within each class, but not independent*; notebook 2, section 3; handout section 4.3.

### Your prediction, before you run the next cell

Near zero, the raw correlation says independent. Will the squares say the same for faulty pumps?

In [9]:
far = ((X - X.mean()) / X.std()) ** 2
for c, name in ((0, "healthy"), (1, "faulty")):
    print(f"{name:<8} readings {X[y == c].corr().iloc[0, 1]:+.3f}   squares {far[y == c].corr().iloc[0, 1]:+.3f}")
fv, fp = far.loc[y == 1, "vibration_hz"], far.loc[y == 1, "pressure_bar"]
low, high = fv.quantile([0.25, 0.75])
print(f"faulty, vibration near the centre: mean pressure square {fp[fv < low].mean():.2f}; "
      f"vibration far out: {fp[fv > high].mean():.2f}")

healthy  readings -0.006   squares +0.168
faulty   readings -0.049   squares -0.420
faulty, vibration near the centre: mean pressure square 2.38; vibration far out: 0.64


**What to look at.** Within each class the readings are uncorrelated, −0.006 and −0.049. Their squared distances from the centre are not: −0.420 among the faulty pumps. Faulty pumps central in vibration have pressure far out (mean square 2.38), those far out in vibration have it near the centre (0.64): knowing one reading changes what to expect of the other, so the independence Naive Bayes assumes is false.

**Try changing.** leave out the flipped labels (`_standardised_radius(...) > 1` against `y`): what happens to the healthy +0.168?

## 9. The interaction: each sensor alone says nothing

Sensors A and B: faulty when exactly one is high. Naive Bayes sees one sensor at a time; give it the product of the two and see what changes.

*From:* slides *A second dataset: the signal is an interaction*; *0.404: below the baseline, and below chance*; notebook 2, section 4.

### Your prediction, before you run the next cell

Naive Bayes scores 0.404 here. With a third column, A times B, will it stay below chance or jump towards k-NN's 0.967?

In [10]:
Xi, yi = load_interacting()
print(f"{len(Xi)} pumps, {int(yi.sum())} faulty; majority baseline {max(yi.mean(), 1 - yi.mean()):.3f}")
nb_pipe = make_pipeline(StandardScaler(), GaussianNB())
print(f"Naive Bayes on A and B:          {cross_val_score(nb_pipe, Xi, yi, cv=folds).mean():.3f}")
with_product = Xi.assign(a_times_b=Xi["sensor_a"] * Xi["sensor_b"])
print(f"Naive Bayes on A, B and A x B:   {cross_val_score(nb_pipe, with_product, yi, cv=folds).mean():.3f}")

1200 pumps, 628 faulty; majority baseline 0.523
Naive Bayes on A and B:          0.404
Naive Bayes on A, B and A x B:   0.972


**What to look at.** On A and B, Naive Bayes scores 0.404, below the 0.523 baseline: each sensor alone carries no information about the class. Give it the product A × B, a column that is positive when the two agree and negative when they differ, and it scores 0.972: the interaction is now one column, which a one-reading-at-a-time model can use.

**Try changing.** give it only the product, without A and B: what does it score?

## 10. Confidence when right, confidence when wrong

Fit on 900 interacting pumps, test on 300. Confidence is the probability given to the class predicted, `max(P, 1 - P)`.

*From:* slide *Its probabilities cannot be trusted*; notebook 2, section 5; handout section 4.5.

### Your prediction, before you run the next cell

Will the model be more confident when it is right than when it is wrong? And is its accuracy close to its mean confidence?

In [11]:
Xtr, Xte, ytr, yte = train_test_split(Xi, yi, test_size=0.25, random_state=0, stratify=yi)
proba = make_pipeline(StandardScaler(), GaussianNB()).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
right = (proba >= 0.5).astype(int) == yte.to_numpy()
confidence = np.maximum(proba, 1 - proba)
print(f"confidence when right {confidence[right].mean():.3f}, when wrong {confidence[~right].mean():.3f}")
print(f"right on {right.sum()} of {len(right)} = {right.mean():.3f}; mean confidence {confidence.mean():.3f}")

confidence when right 0.567, when wrong 0.555
right on 133 of 300 = 0.443; mean confidence 0.560


**What to look at.** Confidence when right, 0.567, and when wrong, 0.555, are almost the same: the model's probability gives no way to tell its right answers from its wrong ones. And it is right on 133 of 300, 0.443, while its mean confidence is 0.560: it says 56% and is right 44% of the time, which is what *not calibrated* means.

**Try changing.** use the product column of atom 9: are the two confidences still alike?

## 11. The widest margin, in one dimension

Healthy pumps at 1, 2 and 3, faulty at 7, 8 and 9. Every threshold between 3 and 7 makes no error; a linear SVM picks one.

*From:* slide *SVM: among the lines that fit, take the widest margin*; handout section 5.1.

### Your prediction, before you run the next cell

Where will the SVM put its threshold: next to the healthy pumps, next to the faulty ones, or in the middle? Which points will be support vectors?

In [12]:
x1 = np.array([[1.0], [2.0], [3.0], [7.0], [8.0], [9.0]])
y1 = np.array([0, 0, 0, 1, 1, 1])
line = SVC(kernel="linear", C=1_000).fit(x1, y1)
w, b = line.coef_[0, 0], line.intercept_[0]
print(f"w = {w:.3f}, b = {b:.3f}: threshold at x = {-b / w:.3f}")
print(f"support vectors: {line.support_vectors_.ravel()};  slab width 2/|w| = {2 / abs(w):.3f}")
print(f"a new pump at 4 is called {'faulty' if line.predict([[4.0]])[0] else 'healthy'}")

w = 0.500, b = -2.500: threshold at x = 5.000
support vectors: [3. 7.];  slab width 2/|w| = 4.000
a new pump at 4 is called healthy


**What to look at.** The SVM puts its threshold at 5, in the middle of the empty gap, 2 from each side; the support vectors are the two pumps on the edges, 3 and 7, and the slab is 2/|w| = 4 wide. A new healthy pump at 4 is called healthy, which a threshold at 3.5, with the same zero errors, would get wrong.

**Try changing.** move the faulty pump at 7 to 6: where does the threshold go?

## 12. Slack, the hinge, and the weights

A soft-margin linear SVM, C = 1, on the six pumps plus a healthy one at 8.5, deep among the faulty ones. Each point's slack is `max(0, 1 - y f(x))` with y = −1 or +1; `dual_coef_` holds each support vector's weight times its label.

*From:* slides *Not separable? Charge for the violations*; *Three kinds of point*; *The hinge loss stops at zero*; *The SVM's solution: a weighted sum of pumps*.

### Your prediction, before you run the next cell

Which points will have slack above zero? And which weights will sit at the ceiling C = 1?

In [13]:
x2 = np.vstack([x1, [[8.5]]])
y2 = np.r_[y1, 0]
soft = SVC(kernel="linear", C=1).fit(x2, y2)
f = soft.decision_function(x2)
slack = np.maximum(0, 1 - (2 * y2 - 1) * f)
weight = np.zeros(len(x2)); weight[soft.support_] = np.abs(soft.dual_coef_[0])
print(pd.DataFrame({"x": x2.ravel(), "label": y2, "f(x)": f.round(3),
                    "y f(x)": ((2 * y2 - 1) * f).round(3), "slack": slack.round(3),
                    "weight a": weight.round(3)}).to_string(index=False))
rebuilt = (soft.dual_coef_[0] * soft.support_vectors_[:, 0]).sum()
print(f"\nw from coef_ {soft.coef_[0, 0]:.4f}; w rebuilt as sum of a y x {rebuilt:.4f}")

  x  label  f(x)  y f(x)  slack  weight a
1.0      0  -1.8     1.8    0.0      0.00
2.0      0  -1.4     1.4    0.0      0.00
3.0      0  -1.0     1.0    0.0      0.38
7.0      1   0.6     0.6    0.4      1.00
8.0      1   1.0     1.0    0.0      0.38
9.0      1   1.4     1.4    0.0      0.00
8.5      0   1.2    -1.2    2.2      1.00

w from coef_ 0.4000; w rebuilt as sum of a y x 0.4000


**What to look at.** The healthy pump at 8.5 sits on the wrong side, y f(x) = −1.2, slack 2.2; the faulty pump at 7 is inside the slab, y f(x) = 0.6, slack 0.4; both carry the ceiling weight C = 1. The pumps at 3 and 8 sit exactly on the edges, y f(x) = 1, with weights between 0 and C (0.38). The rest are outside the slab, slack 0 and weight 0. And w, 0.4, is rebuilt exactly as the sum of weight × label × x over the support vectors.

**Try changing.** C = 100: does the threshold move, and what happens to the slab's width?

## 13. The kernel trick by hand, and the RBF similarity

Two points u = (1, 2) and v = (3, 1): lift both with φ(x) = (x₁², √2·x₁x₂, x₂²) and multiply, or square their ordinary inner product. Then `rbf_kernel` at one, two and three standard deviations.

*From:* slides *So the new coordinates are never computed*; *The kernel trick by hand: 25 = 25*; *The RBF kernel: similarity falls with distance*.

### Your prediction, before you run the next cell

Will the two routes give the same number? And with γ = 0.5, is the similarity at one standard deviation nearer 0.9, 0.6 or 0.3?

In [14]:
u, v = np.array([1.0, 2.0]), np.array([3.0, 1.0])
phi = lambda point: np.array([point[0] ** 2, np.sqrt(2) * point[0] * point[1], point[1] ** 2])
print(f"phi(u) . phi(v) = {phi(u) @ phi(v):.1f};  (u . v)^2 = {(u @ v) ** 2:.1f}")
for d in (1, 2, 3):
    print(f"RBF similarity at {d} sd, gamma 0.5: {rbf_kernel([[0.0, 0.0]], [[d, 0.0]], gamma=0.5)[0, 0]:.3f}")
print(f"it halves at {np.sqrt(np.log(2) / 0.5):.2f} sd")

phi(u) . phi(v) = 25.0;  (u . v)^2 = 25.0
RBF similarity at 1 sd, gamma 0.5: 0.607
RBF similarity at 2 sd, gamma 0.5: 0.135
RBF similarity at 3 sd, gamma 0.5: 0.011
it halves at 1.18 sd


**What to look at.** Lifting and multiplying gives 25; squaring the ordinary inner product, 5, also gives 25: the kernel (u · v)² is the inner product in the lifted space, computed without lifting. The RBF similarity is 0.607 at one standard deviation, 0.135 at two and 0.011 at three, and it halves at 1.18.

**Try changing.** gamma = 50: at what distance does the similarity halve now?

## 14. The 48 Hz pump's vote, by hand

The RBF SVM on the standardised pumps: weight times label times similarity, summed over the support vectors, plus b, against `decision_function`.

*From:* slides *Training: one weight per pump*; *The SVM's score at 48 Hz, 5.6 bar*; *One pump's vote, drawn*; notebook 3, section 5.

### Your prediction, before you run the next cell

Will the sum by hand equal `decision_function` exactly? Is the score above or below 1, the edge of the slab?

In [15]:
svm_pipe = make_pipeline(StandardScaler(), SVC()).fit(X, y)
svm = svm_pipe[-1]
gamma = 1 / (2 * svm_pipe[0].transform(X).var())                 # what gamma="scale" gives
weights = np.abs(svm.dual_coef_[0])
print(f"{len(svm.support_)} support vectors: {int(np.isclose(weights, svm.C).sum())} at C, "
      f"{int((~np.isclose(weights, svm.C)).sum())} below; {len(X) - len(svm.support_)} pumps at weight 0")
z = svm_pipe[0].transform(pump)
similarity = rbf_kernel(svm.support_vectors_, z, gamma=gamma)[:, 0]
votes = svm.dual_coef_[0] * similarity
print(f"faulty votes {votes[votes > 0].sum():+.3f}, healthy votes {votes[votes < 0].sum():+.3f}, "
      f"b {svm.intercept_[0]:+.3f}")
print(f"by hand {votes.sum() + svm.intercept_[0]:+.3f};  decision_function {svm_pipe.decision_function(pump)[0]:+.3f}")

278 support vectors: 268 at C, 10 below; 922 pumps at weight 0
faulty votes +37.455, healthy votes -36.677, b +1.105
by hand +1.883;  decision_function +1.883


**What to look at.** 278 support vectors: 268 at the ceiling C, 10 below it; the other 922 pumps have weight 0. The faulty votes add to +37.455, the healthy ones to −36.677; with b = +1.105 the total is +1.883, exactly `decision_function`. It is above 1, so the pump sits beyond the faulty edge of the slab.

**Try changing.** the pump at the design point, 42 Hz and 5.6 bar: what is its score, and its sign?

## 15. γ and C: training against cross-validated

Three settings of the RBF SVM, both scores from the same five folds, and the number of support vectors each keeps.

*From:* slides *Training against cross-validated, three settings*; *Overfitting you can see*; notebook 3, section 6.

### Your prediction, before you run the next cell

γ = 50 with C = 1000: will its training score be the highest of the three? Its cross-validated score?

In [16]:
for gamma_, C in ((0.1, 1), (1, 1), (50, 1000)):
    pipe = make_pipeline(StandardScaler(), SVC(gamma=gamma_, C=C))
    result = cross_validate(pipe, X, y, cv=folds, return_train_score=True)
    n_sv = len(pipe.fit(X, y)[-1].support_)
    print(f"gamma {gamma_:>4}, C {C:>4}: training {result['train_score'].mean():.3f}, "
          f"cross-validated {result['test_score'].mean():.3f}, {n_sv} support vectors")

gamma  0.1, C    1: training 0.934, cross-validated 0.929, 420 support vectors


gamma    1, C    1: training 0.948, cross-validated 0.944, 266 support vectors


gamma   50, C 1000: training 0.996, cross-validated 0.902, 548 support vectors


**What to look at.** γ = 50 with C = 1000 has the highest training score, 0.996, and the lowest cross-validated one, 0.902: overfitting. It also keeps 548 support vectors against 266 at γ = 1, C = 1: every flipped pump gets an island and becomes a support vector. The middle setting, 0.948 and 0.944, is the one to want.

**Try changing.** γ = 50 with C = 1, then γ = 1 with C = 1000: which knob, alone, does the damage?

## What to take away

- **k-NN stores and votes**: `fit` keeps the pumps, `kneighbors` finds the voters,
  and the scaler decides who they are.
- **A training score at k = 1 is 1.000 by construction**; measured inside the folds it
  tells you where the curves meet.
- **Distances stop varying** as columns are added, and the test that compares class
  means cannot find the columns that matter on a ring.
- **Naive Bayes is ten numbers and a product**, best read as three ratios or one gap
  between two log totals; its probabilities are not to be trusted.
- **An SVM is a weighted vote of its support vectors**: `dual_coef_` holds the
  weights, the kernel the similarities, and the sum is `decision_function`.